# Data Loading

This notebook builds a knowledge graph from SEC 10-K filing data. You will load structured and unstructured data, create entity nodes, split filing text into chunks, and store everything in Neo4j with relationships that connect the structured and unstructured layers.

**Learning Objectives:**
- Understand the two-layer graph pattern: structured entities + unstructured document chunks
- Create Company, Product, and RiskFactor nodes from structured data
- Create Document and Chunk nodes from filing text
- Link the layers with FILED, OFFERS, FACES_RISK, and FROM_CHUNK relationships

## The Two-Layer Graph Pattern

GraphRAG applications combine **structured** knowledge (entities and relationships) with **unstructured** text (document chunks). This notebook builds both layers and connects them.

**Structured layer** — entities from the filing:
```
(:Company)-[:OFFERS]->(:Product)
(:Company)-[:FACES_RISK]->(:RiskFactor)
```

**Unstructured layer** — filing text split into retrievable chunks:
```
(:Document)<-[:FROM_DOCUMENT]-(:Chunk)-[:NEXT_CHUNK]->(:Chunk)
```

**Cross-links** connect the layers:
- **FILED** links the Company to its Document: `(:Company)-[:FILED]->(:Document)`
- **FROM_CHUNK** links Products to the Chunks that mention them: `(:Product)-[:FROM_CHUNK]->(:Chunk)`

These cross-links are what make GraphRAG powerful. A vector search finds relevant chunks, then graph traversal reaches the structured entities — giving the LLM both the raw text and the knowledge graph context.

In [ ]:
%pip install "neo4j-graphrag[bedrock] @ git+https://github.com/neo4j-partners/neo4j-graphrag-python.git@bedrock-embeddings" -q

In [ ]:
import json
import os

from lib.data_utils import split_text
from neo4j import GraphDatabase
from dotenv import load_dotenv

# Load configuration
load_dotenv('../CONFIG.txt')

NEO4J_URI = os.getenv('NEO4J_URI')
NEO4J_USERNAME = os.getenv('NEO4J_USERNAME')
NEO4J_PASSWORD = os.getenv('NEO4J_PASSWORD')

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USERNAME, NEO4J_PASSWORD))
driver.verify_connectivity()
print('Connected to Neo4j successfully!')

## Clear Existing Data

Start with a clean graph by removing any nodes and relationships from previous runs.

In [ ]:
def clear_graph(driver):
    """Remove all nodes and relationships from the database."""
    with driver.session() as session:
        result = session.run("""
            MATCH (n)
            DETACH DELETE n
            RETURN count(n) as deleted
        """)
        count = result.single()['deleted']
        print(f'Deleted {count} nodes')

clear_graph(driver)

## Load Financial Filing Data

Load the structured filing data from `financial_data.json`. This file contains Apple's 10-K annual report data organized into:
- **Company** metadata (name, ticker, CIK)
- **Products** mentioned in the filing
- **Risk factors** identified in the filing
- **Filing text** — the full unstructured text for chunking

In [ ]:
with open('financial_data.json', 'r') as f:
    filing_data = json.load(f)

filing_text = filing_data['filing_text']

print(f'Company: {filing_data["company"]["name"]} ({filing_data["company"]["ticker"]})')
print(f'Products: {len(filing_data["products"])}')
print(f'Risk Factors: {len(filing_data["risk_factors"])}')
print(f'Filing text: {len(filing_text)} characters')
print(f'\nPreview (first 300 chars):\n{filing_text[:300]}...')

## Create Structured Layer

Create the Company node, then the Product and RiskFactor nodes with their relationships. This is the structured knowledge graph layer — facts from the filing that retrieval queries can traverse to enrich search results.

In [ ]:
def create_company(driver, company: dict):
    """Create a Company node with name, ticker, and CIK."""
    with driver.session() as session:
        result = session.run("""
            MERGE (c:Company {name: $name})
            SET c.ticker = $ticker, c.cik = $cik
            RETURN elementId(c) AS company_id
        """, name=company['name'], ticker=company['ticker'], cik=company['cik'])
        company_id = result.single()['company_id']
        print(f'Created Company: {company["name"]} ({company["ticker"]})')
        return company_id


def create_products(driver, company_id: str, products: list[dict]):
    """Create Product nodes and OFFERS relationships from Company."""
    with driver.session() as session:
        session.run("""
            MATCH (c:Company) WHERE elementId(c) = $company_id
            UNWIND $products AS prod
            MERGE (p:Product {name: prod.name})
            SET p.description = prod.description
            MERGE (c)-[:OFFERS]->(p)
        """, company_id=company_id, products=products)
        print(f'Created {len(products)} Product nodes with OFFERS relationships')


def create_risk_factors(driver, company_id: str, risk_factors: list[dict]):
    """Create RiskFactor nodes and FACES_RISK relationships from Company."""
    with driver.session() as session:
        session.run("""
            MATCH (c:Company) WHERE elementId(c) = $company_id
            UNWIND $risks AS risk
            MERGE (r:RiskFactor {name: risk.name})
            SET r.description = risk.description
            MERGE (c)-[:FACES_RISK]->(r)
        """, company_id=company_id, risks=risk_factors)
        print(f'Created {len(risk_factors)} RiskFactor nodes with FACES_RISK relationships')


company_id = create_company(driver, filing_data['company'])
create_products(driver, company_id, filing_data['products'])
create_risk_factors(driver, company_id, filing_data['risk_factors'])

## Create Document and Link to Company

Create the Document node for the filing and connect it to the Company with a `FILED` relationship. This cross-link bridges the structured and unstructured layers — retrieval queries traverse this path from chunks to entities:

```
(:Chunk)-[:FROM_DOCUMENT]->(:Document)<-[:FILED]-(:Company)
```

In [ ]:
def create_document(driver, company_id: str, document: dict):
    """Create a Document node and FILED relationship from Company."""
    with driver.session() as session:
        result = session.run("""
            MATCH (c:Company) WHERE elementId(c) = $company_id
            MERGE (d:Document {name: $name})
            SET d.source = $source
            MERGE (c)-[:FILED]->(d)
            RETURN elementId(d) AS doc_id
        """, company_id=company_id, name=document['name'], source=document['source'])
        doc_id = result.single()['doc_id']
        print(f'Created Document: {document["name"]}')
        print(f'Created FILED relationship: {filing_data["company"]["name"]} -> {document["name"]}')
        return doc_id

doc_id = create_document(driver, company_id, filing_data['document'])

## Split Text into Chunks

Split the filing text into smaller chunks for embedding and retrieval. Each chunk is 500 characters with 50 characters of overlap between consecutive chunks. The overlap ensures that sentences split across chunk boundaries still appear in at least one complete chunk.

In [ ]:
chunks = split_text(filing_text)
print(f'Split into {len(chunks)} chunks\n')
for i, chunk in enumerate(chunks):
    print(f'Chunk {i}: {len(chunk)} chars')
    print(f'  {chunk[:80]}...\n')

## Create Chunk Nodes and Link to Document

Create a Chunk node for each piece of text and connect it to the Document node with a `FROM_DOCUMENT` relationship. Then link consecutive chunks with `NEXT_CHUNK` relationships to preserve reading order.

In [ ]:
def create_chunks(driver, doc_id: str, chunks: list[str]):
    """Create Chunk nodes linked to a Document via FROM_DOCUMENT."""
    chunk_data = [{"index": i, "text": t} for i, t in enumerate(chunks)]
    with driver.session() as session:
        session.run("""
            MATCH (d:Document) WHERE elementId(d) = $doc_id
            UNWIND $chunks AS chunk
            MERGE (c:Chunk {index: chunk.index})
            SET c.text = chunk.text
            MERGE (c)-[:FROM_DOCUMENT]->(d)
        """, doc_id=doc_id, chunks=chunk_data)
        print(f'Created {len(chunks)} Chunk nodes with FROM_DOCUMENT relationships')


def link_chunks(driver, num_chunks: int):
    """Create NEXT_CHUNK relationships between consecutive chunks."""
    pairs = [{"idx1": i, "idx2": i + 1} for i in range(num_chunks - 1)]
    with driver.session() as session:
        session.run("""
            UNWIND $pairs AS pair
            MATCH (c1:Chunk {index: pair.idx1})
            MATCH (c2:Chunk {index: pair.idx2})
            MERGE (c1)-[:NEXT_CHUNK]->(c2)
        """, pairs=pairs)
        print(f'Created {num_chunks - 1} NEXT_CHUNK relationships')


create_chunks(driver, doc_id, chunks)
link_chunks(driver, len(chunks))

## Link Products to Chunks

Create `FROM_CHUNK` relationships that connect each Product to the Chunk(s) that mention it. This enables retrieval queries to find which products are discussed in a matched chunk. The matching uses simple substring containment — the product name must appear in the chunk text.

In [ ]:
def link_products_to_chunks(driver, products: list[dict]):
    """Create FROM_CHUNK relationships from Products to Chunks that mention them."""
    total = 0
    with driver.session() as session:
        for product in products:
            result = session.run("""
                MATCH (p:Product {name: $name})
                MATCH (c:Chunk)
                WHERE c.text CONTAINS $name
                MERGE (p)-[:FROM_CHUNK]->(c)
                RETURN count(*) AS linked
            """, name=product['name'])
            count = result.single()['linked']
            if count > 0:
                print(f'  {product["name"]} -> {count} chunk(s)')
                total += count
    print(f'\nCreated {total} FROM_CHUNK relationships')

link_products_to_chunks(driver, filing_data['products'])

## Verify Graph Structure

Query the graph to confirm that all nodes and relationships were created correctly.

In [ ]:
def show_graph_structure(driver):
    """Display a summary of the graph structure."""
    with driver.session() as session:
        # Count nodes by label
        node_result = session.run("""
            MATCH (n)
            WITH labels(n)[0] AS label, n
            WHERE label IS NOT NULL
            RETURN label, count(n) AS count
            ORDER BY label
        """)
        print('=== Node Counts ===')
        for record in node_result:
            print(f'  {record["label"]}: {record["count"]}')

        # Count relationships by type
        rel_result = session.run("""
            MATCH ()-[r]->()
            RETURN type(r) AS type, count(r) AS count
            ORDER BY type
        """)
        print('\n=== Relationship Counts ===')
        for record in rel_result:
            print(f'  {record["type"]}: {record["count"]}')

        # Show structured layer
        struct_result = session.run("""
            MATCH (c:Company)
            OPTIONAL MATCH (c)-[:OFFERS]->(p:Product)
            OPTIONAL MATCH (c)-[:FACES_RISK]->(r:RiskFactor)
            OPTIONAL MATCH (c)-[:FILED]->(d:Document)
            RETURN c.name AS company, c.ticker AS ticker,
                   count(DISTINCT p) AS products,
                   count(DISTINCT r) AS risks,
                   count(DISTINCT d) AS documents
        """)
        print('\n=== Structured Layer ===')
        for record in struct_result:
            print(f'  {record["company"]} ({record["ticker"]})')
            print(f'    Products: {record["products"]}, Risk Factors: {record["risks"]}, Documents: {record["documents"]}')

        # Show chunk chain
        chain_result = session.run("""
            MATCH (c:Chunk)
            OPTIONAL MATCH (c)-[:NEXT_CHUNK]->(next:Chunk)
            OPTIONAL MATCH (p:Product)-[:FROM_CHUNK]->(c)
            RETURN c.index AS idx, next.index AS next_idx,
                   left(c.text, 60) AS preview,
                   collect(DISTINCT p.name) AS products
            ORDER BY c.index
        """)
        print('\n=== Chunk Chain ===')
        for record in chain_result:
            next_str = f' -> Chunk {record["next_idx"]}' if record['next_idx'] is not None else ' (end)'
            products_str = f' [Products: {", ".join(record["products"])}]' if record['products'] else ''
            print(f'  Chunk {record["idx"]}: "{record["preview"]}..."{next_str}{products_str}')

show_graph_structure(driver)

## Summary

You built a two-layer knowledge graph from SEC 10-K filing data:

**Structured layer:**
1. **Company node** stores the filing entity (name, ticker, CIK)
2. **Product nodes** represent products and services the company offers
3. **RiskFactor nodes** capture risk factors from the filing
4. **OFFERS** and **FACES_RISK** relationships connect the company to its products and risks

**Unstructured layer:**
5. **Document node** stores filing metadata (name and source)
6. **Chunk nodes** hold individual text segments from the filing
7. **FROM_DOCUMENT** relationships link every chunk to its source document
8. **NEXT_CHUNK** relationships preserve the original reading order

**Cross-links:**
9. **FILED** connects the Company to its Document
10. **FROM_CHUNK** connects Products to the Chunks that mention them

This graph structure is the foundation for everything that follows. In the next notebook, you will generate vector embeddings for each chunk so the graph supports semantic search.

---

**Next:** [Embeddings](02_embeddings.ipynb)

In [ ]:
# Cleanup
driver.close()
print('Connection closed.')